# Task 1 composer classification via REMI Transformer (Colab GPU)

Tokenizes the MIDIs with REMI (via `miditok`), trains a small BERT encoder from scratch with transposition augmentation, and writes `predictions1.json`. The previous feature-ensemble baseline scored 0.6633 on the leaderboard with a 0.24 gap to CV; this notebook aims to close that by letting the model learn its own transposition-invariant features instead of hand-rolling them.

**Before running:**
1. Upload `student_files_updated.zip` to your Google Drive at `/content/drive/MyDrive/CSE153/student_files_updated.zip`.
2. Runtime menu, set GPU runtime (A100 or V100 preferred).
3. Run the cells top to bottom.

Saves `predictions1.json` to the Colab working directory and to your Drive at `/content/drive/MyDrive/CSE153/predictions1.json`.

## 1. GPU check and dependency install

In [ ]:
!nvidia-smi

In [ ]:
# miditok handles REMI tokenization of MIDI files
# transformers provides the BertModel / BertConfig classes we use as the encoder
!pip -q install miditok transformers
import torch, numpy as np, transformers, miditok
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('miditok:', miditok.__version__, 'transformers:', transformers.__version__)

## 2. Mount Drive and unzip the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
ZIP_PATH = '/content/drive/MyDrive/CSE153/student_files_updated.zip'
WORKDIR = '/content/work'
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}; upload it to Drive or change ZIP_PATH'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
if not os.path.isdir('student_files'):
    !unzip -q -o {ZIP_PATH} -d {WORKDIR}
    if os.path.isdir(os.path.join(WORKDIR, '__MACOSX')):
        shutil.rmtree(os.path.join(WORKDIR, '__MACOSX'))
print('Contents:', os.listdir('student_files'))
print('Train MIDIs:', len(os.listdir('student_files/task1_composer_classification/midis')))

## 3. Tokenize every MIDI with REMI

REMI represents each note as Bar / Position / Pitch / Velocity / Duration tokens. Tokenization runs once over the whole training plus test set and is cached, so a notebook restart skips this step.

In [ ]:
import os, json, random, numpy as np, torch
from tqdm.auto import tqdm
from miditok import REMI, TokenizerConfig

DATAROOT = 'student_files/task1_composer_classification'
TOKEN_CACHE = '/content/drive/MyDrive/CSE153/task1_tokens.npz'

tok_config = TokenizerConfig(
    pitch_range=(21, 109),          # piano range, MIDI 21 = A0, 108 = C8
    beat_res={(0, 4): 8, (4, 12): 4},
    num_velocities=32,
    use_chords=False,
    use_rests=False,
    use_tempos=True,
    num_tempos=32,
    tempo_range=(40, 250),
    use_time_signatures=True,
    special_tokens=['PAD', 'BOS', 'EOS', 'MASK'],
)
tokenizer = REMI(tok_config)
VOCAB_SIZE = len(tokenizer)
print('vocab size:', VOCAB_SIZE)
PAD_ID = tokenizer['PAD_None']
BOS_ID = tokenizer['BOS_None']
print('PAD id:', PAD_ID, 'BOS id:', BOS_ID)

In [ ]:
def midi_to_ids(path):
    seq = tokenizer(os.path.join(DATAROOT, path))
    if isinstance(seq, list):
        ids = []
        for s in seq:
            ids.extend(s.ids)
    else:
        ids = list(seq.ids)
    return np.array(ids, dtype=np.int64)

train_meta = eval(open(os.path.join(DATAROOT, 'train.json')).read())
test_list  = eval(open(os.path.join(DATAROOT, 'test.json')).read())
train_paths = list(train_meta.keys())
train_labels = np.array([int(train_meta[p]) for p in train_paths])
test_paths = list(test_list)

if os.path.exists(TOKEN_CACHE):
    print('loading cached tokens from', TOKEN_CACHE)
    cache = np.load(TOKEN_CACHE, allow_pickle=True)
    tokens = {str(k): v for k, v in cache['tokens'].item().items()}
else:
    tokens = {}
    for p in tqdm(train_paths + test_paths, desc='tokenizing'):
        tokens[p] = midi_to_ids(p)
    np.savez_compressed(TOKEN_CACHE, tokens=tokens)
    print('saved', TOKEN_CACHE)

lens = np.array([len(tokens[p]) for p in train_paths])
print(f'token-seq length: min={lens.min()} median={int(np.median(lens))} max={lens.max()} mean={lens.mean():.0f}')

## 4. Transposition map

Augmentation is done in token space: for a given semitone offset, every `Pitch_X` token is remapped to `Pitch_(X+offset)` if that pitch is in the vocabulary. All other tokens (Bar, Position, Velocity, Duration, etc) are unchanged.

In [ ]:
TRANSPOSITIONS = list(range(-5, 7))  # -5..+6 semitones

def make_transpose_map(semitones):
    mp = np.arange(VOCAB_SIZE, dtype=np.int64)
    for tok_str, tok_id in tokenizer.vocab.items():
        if tok_str.startswith('Pitch_'):
            try:
                pitch = int(tok_str.split('_')[1])
            except ValueError:
                continue
            target = f'Pitch_{pitch + semitones}'
            if target in tokenizer.vocab:
                mp[tok_id] = tokenizer.vocab[target]
    return mp

TRANSPOSE_MAPS = {s: make_transpose_map(s) for s in TRANSPOSITIONS}
print('built transpose maps for', list(TRANSPOSE_MAPS.keys()))

## 5. Datasets, model, helpers

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertConfig, BertModel
from sklearn.model_selection import train_test_split
from collections import Counter, defaultdict

MAX_LEN = 512
STRIDE  = 256
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 0.01
N_CLASSES = 8
WINDOWS_PER_TRAIN_MIDI = 4
INFER_WINDOWS_PER_MIDI = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(0); np.random.seed(0); random.seed(0)

In [ ]:
def pack_window(window_ids):
    # prepend BOS to act as a classification token, then right-pad to MAX_LEN
    inner = window_ids[: MAX_LEN - 1]
    ids = np.full(MAX_LEN, PAD_ID, dtype=np.int64)
    ids[0] = BOS_ID
    ids[1:1 + len(inner)] = inner
    attn = np.zeros(MAX_LEN, dtype=np.int64)
    attn[: 1 + len(inner)] = 1
    return ids, attn

def enumerate_windows(tok_ids, max_len=MAX_LEN, stride=STRIDE):
    inner = max_len - 1
    L = len(tok_ids)
    if L <= inner:
        return [tok_ids]
    starts = list(range(0, L - inner + 1, stride))
    if starts[-1] + inner < L:
        starts.append(L - inner)
    return [tok_ids[s:s + inner] for s in starts]

class TrainSet(Dataset):
    """Each __getitem__ samples a random window from a random MIDI with a random transposition."""
    def __init__(self, tokens_list, labels, n_per_midi=WINDOWS_PER_TRAIN_MIDI):
        self.tokens = tokens_list
        self.labels = labels
        self.n = n_per_midi
    def __len__(self):
        return len(self.tokens) * self.n
    def __getitem__(self, idx):
        midi_idx = idx % len(self.tokens)
        tok = self.tokens[midi_idx]
        semi = random.choice(TRANSPOSITIONS)
        tok = TRANSPOSE_MAPS[semi][tok]
        L = len(tok)
        inner = MAX_LEN - 1
        if L <= inner:
            window = tok
        else:
            start = random.randint(0, L - inner)
            window = tok[start:start + inner]
        ids, attn = pack_window(window)
        return torch.from_numpy(ids), torch.from_numpy(attn), int(self.labels[midi_idx])

class WindowedEvalSet(Dataset):
    """Enumerates every window for every MIDI; aggregation by path is done at scoring time."""
    def __init__(self, tokens_list, paths):
        self.items = []
        for tok, p in zip(tokens_list, paths):
            for w in enumerate_windows(tok):
                self.items.append((w, p))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        window, p = self.items[idx]
        ids, attn = pack_window(window)
        return torch.from_numpy(ids), torch.from_numpy(attn), p

In [ ]:
class RemiBert(nn.Module):
    """Small BERT encoder over REMI tokens, mean-pooled into a composer logits head."""
    def __init__(self, vocab_size, n_classes, max_len=MAX_LEN,
                 d_model=256, n_layers=6, n_heads=8, ff=1024, dropout=0.1):
        super().__init__()
        cfg = BertConfig(
            vocab_size=vocab_size,
            hidden_size=d_model,
            num_hidden_layers=n_layers,
            num_attention_heads=n_heads,
            intermediate_size=ff,
            max_position_embeddings=max_len,
            type_vocab_size=1,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            pad_token_id=PAD_ID,
        )
        self.bert = BertModel(cfg)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(d_model, n_classes)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state                                  # (B, T, D)
        mask = attention_mask.unsqueeze(-1).float()                   # (B, T, 1)
        pooled = (last * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        return self.head(self.dropout(pooled))                        # logits (B, C)

In [ ]:
def aggregate_predictions(model, loader):
    """Run the model over every window, then average logits per source MIDI path."""
    model.eval()
    logits_by_path = defaultdict(list)
    with torch.no_grad():
        for ids, attn, paths in loader:
            ids = ids.to(DEVICE); attn = attn.to(DEVICE)
            logits = model(ids, attn).cpu().numpy()
            for i, p in enumerate(paths):
                logits_by_path[p].append(logits[i])
    return {p: np.mean(np.stack(ls), axis=0) for p, ls in logits_by_path.items()}

def eval_accuracy(model, loader, label_lookup):
    avg = aggregate_predictions(model, loader)
    correct = 0; total = 0
    for p, l in avg.items():
        if int(np.argmax(l)) == int(label_lookup[p]):
            correct += 1
        total += 1
    return correct / max(total, 1)

## 6. Stratified split, loaders, optimizer

In [ ]:
tr_idx, va_idx = train_test_split(
    np.arange(len(train_paths)), test_size=0.1, random_state=0, stratify=train_labels
)
tr_tokens = [tokens[train_paths[i]] for i in tr_idx]
tr_labels = train_labels[tr_idx]
va_tokens = [tokens[train_paths[i]] for i in va_idx]
va_paths  = [train_paths[i] for i in va_idx]
va_labels = train_labels[va_idx]
te_tokens = [tokens[p] for p in test_paths]
print('train MIDIs:', len(tr_tokens), 'val MIDIs:', len(va_tokens), 'test MIDIs:', len(te_tokens))

train_set = TrainSet(tr_tokens, tr_labels)
val_set   = WindowedEvalSet(va_tokens, va_paths)
test_set  = WindowedEvalSet(te_tokens, test_paths)

loader_tr = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
loader_va = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
loader_te = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

val_label_lookup = {p: l for p, l in zip(va_paths, va_labels)}

In [ ]:
model = RemiBert(VOCAB_SIZE, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params/1e6:.2f}M')

# inverse-frequency class weights, scaled so weights average to 1
cls_counts = Counter(int(c) for c in tr_labels)
weights = np.array([1.0 / cls_counts[i] for i in range(N_CLASSES)], dtype=np.float32)
weights = weights / weights.mean()
cls_weight = torch.tensor(weights, device=DEVICE)
print('class counts:', dict(cls_counts))
print('class weights:', weights)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
crit = nn.CrossEntropyLoss(weight=cls_weight)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

## 7. Train

In [ ]:
best_acc = -1.0
best_state = None
for ep in range(EPOCHS):
    model.train()
    running, nb = 0.0, 0
    for ids, attn, y in tqdm(loader_tr, desc=f'Epoch {ep+1}/{EPOCHS}'):
        ids = ids.to(DEVICE, non_blocking=True)
        attn = attn.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        loss = crit(model(ids, attn), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running += loss.item(); nb += 1
    sched.step()
    val_acc = eval_accuracy(model, loader_va, val_label_lookup)
    print(f'[ep{ep+1}] loss={running/nb:.4f} val_acc={val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
print('Best val acc:', best_acc)

## 8. Generate predictions1.json

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_logits = aggregate_predictions(model, loader_te)
preds = {p: int(np.argmax(test_logits[p])) for p in test_paths}

out_local = '/content/work/predictions1.json'
with open(out_local, 'w') as f:
    f.write(repr(preds) + '\n')
print('Wrote', out_local, 'entries=', len(preds))

import shutil
drive_out = '/content/drive/MyDrive/CSE153/predictions1.json'
os.makedirs(os.path.dirname(drive_out), exist_ok=True)
shutil.copy(out_local, drive_out)
print('Saved copy to', drive_out)

In [ ]:
# Sanity check the output format
d = eval(open(out_local).read())
print('len:', len(d))
k = list(d.keys())[0]
print('sample:', repr(k), '->', d[k])
assert all(isinstance(v, int) and 0 <= v < N_CLASSES for v in d.values()), 'malformed predictions'
print('OK')

Download `/content/work/predictions1.json` from the Colab file panel, or pull it from your Drive at `/content/drive/MyDrive/CSE153/predictions1.json`. Drop it in the assignment directory in place of the current one and submit all five files.